# 7. Ablation: no printed prior (E4), and quality against training

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/7_ablation_no_prior.ipynb)

**Runtime: A100.** About 3.5 hours. Run after notebooks 3 to 5, if they finished within their budget.

1. **The ablation**: the same model and schedule as the main run, with the printed word left
   out of the input, trained to the main run's 10% snapshot (`--steps` the main run's total,
   `--stop-at` its 10% step). Resumable.
2. **Scores** on 200 validation writers that the grid did not use (writers 200-399), with the
   setting chosen in notebook 3: the ablation, and the main run's snapshots (10, 25, 50, 75%)
   and final averages, for the comparison at 10% and a figure of quality against training.

Outputs: `WORK/runs/ablation_no_prior/`, `results/e4_*.json`, `WORK/eval/figures/training.png`.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

HF_REC = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # the deployed word recogniser
REC_REVISION = "main"                       # its version is recorded in results/recogniser_source.json
HWD_COMMIT = "eabb5b5"                      # the HWD package (github.com/aimagelab/HWD)

WRITERS = 200
FIRST_WRITER = 200

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

if not os.path.exists(f"{CONTENT}/HWD/setup.py"):
    clone("https://github.com/aimagelab/HWD.git", f"{CONTENT}/HWD", HWD_COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", f"{CONTENT}/HWD", "gudhi")   # gudhi: imported by hwd.scores

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

REC = f"{CONTENT}/recogniser"


def get_recogniser():
    """The deployed recogniser (recogniser.pt, char_lm.pkl, decoding.json) at the version recorded the
    first time (results/recogniser_source.json), so that every notebook scores with the same one."""
    from huggingface_hub import HfApi, snapshot_download
    kept = f"{WORK}/results/recogniser_source.json"
    if os.path.exists(kept):
        source = json.loads(Path(kept).read_text())
    else:
        source = {"repo": HF_REC, "revision": REC_REVISION, "sha": HfApi().model_info(HF_REC, revision=REC_REVISION).sha}
    if not os.path.exists(f"{REC}/source.json"):
        snapshot_download(HF_REC, revision=source["sha"], local_dir=REC,
                          allow_patterns=["recogniser.pt", "char_lm.pkl", "decoding.json", "results.json"])
        Path(f"{REC}/source.json").write_text(json.dumps(source))
    if not os.path.exists(kept):
        shutil.copy(f"{REC}/source.json", kept)
    print("recogniser:", json.loads(Path(f"{REC}/source.json").read_text()))


get_recogniser()

import torch
MAIN, ABL = f"{WORK}/runs/main", f"{WORK}/runs/ablation_no_prior"
GRID = f"{WORK}/results/eval_val_grid.json"
snaps = sorted(Path(MAIN).glob("snap_*.pt"))
if not snaps or not os.path.exists(f"{MAIN}/final.pt") or not os.path.exists(GRID):
    raise SystemExit("the main run's snapshots, final.pt and the validation grid are needed: notebooks 2 and 3")
first = torch.load(snaps[0], map_location="cpu", weights_only=False)
TOTAL, STOP = first["total"], first["step"]
print(f"main run: {TOTAL} steps; snapshots at {[int(p.stem.split('_')[1]) for p in snaps]}; the ablation stops at {STOP}")

## 1. The ablation

In [ ]:
stream(sys.executable, "-u", "scripts/train.py", "--word-repo", WORD, "--data", f"{WORK}/data", "--run", ABL,
       "--local", f"{CONTENT}/ckpt_ablation", "--steps", TOTAL, "--stop-at", STOP, "--no-prior")

## 2. Scores

In [ ]:
jobs = [("ablation_no_prior", f"{ABL}/snap_{STOP:07d}.pt", True)]
jobs += [(f"main_{p.stem}", str(p), False) for p in snaps] + [("main_final", f"{MAIN}/final.pt", False)]
summary = {}
for label, weights, no_prior in jobs:
    out = f"{WORK}/results/e4_{label}.json"
    if not os.path.exists(out):
        stream(sys.executable, "-u", "scripts/evaluate.py", "score", "--word-repo", WORD, "--weights", weights,
               "--data", f"{WORK}/data", "--split", "val", "--writers", WRITERS, "--first-writer", FIRST_WRITER,
               "--setting", GRID, "--recogniser", REC, "--work", f"{CONTENT}/eval/{label}", "--out", out,
               "--label", label, *(["--no-prior"] if no_prior else []))
    r = json.loads(Path(out).read_text())
    step = torch.load(weights, map_location="cpu", weights_only=False).get("step") if weights.endswith(".pt") else None
    summary[label] = {"step": step, **{k: r.get(k) for k in ("greedy_cer", "with_lm_cer", "hwd", "reference_greedy_cer")}}
Path(f"{WORK}/results/e4_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))

In [ ]:
import matplotlib.pyplot as plt

main = sorted((v["step"], v) for k, v in summary.items() if k.startswith("main_"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([s for s, _ in main], [100 * v["greedy_cer"] for _, v in main], "o-", label="main run")
ax[1].plot([s for s, _ in main], [v["hwd"] for _, v in main], "o-", label="main run")
a = summary["ablation_no_prior"]
ax[0].plot([a["step"]], [100 * a["greedy_cer"]], "s", label="no printed prior")
ax[1].plot([a["step"]], [a["hwd"]], "s", label="no printed prior")
ax[0].set_ylabel("greedy CER (%)"); ax[1].set_ylabel("HWD")
for x in ax:
    x.set_xlabel("training step"); x.legend()
fig.tight_layout()
os.makedirs(f"{WORK}/eval/figures", exist_ok=True)
fig.savefig(f"{WORK}/eval/figures/training.png", dpi=150)
plt.show()

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")